# 04 — Memory: short-term context, long-term DuckDB

**Goal.** The loop in nb 03 forgets everything between questions. This notebook shows the two practical kinds of agent memory:

- **Short-term** = the message list. Lives in RAM, fits in the model's context window. Used during a conversation.
- **Long-term** = DuckDB. Persists across runs. Used to remember facts, briefs, and decisions the agent has produced.

**Why split them.** A common mistake in agentic systems is shoving everything into the model's context. Local 8B models break under that load. The right pattern is *stateless agent + stateful tools*: the model carries minimal context, and anything important goes into a tool-accessible store.

**What we'll build.**

1. A new `agent_memory` table in the same DuckDB the rest of the project uses.
2. Two new tools — `remember_note` (write) and `recall_notes` (read) — wired into the registry from nb 01.
3. A multi-turn loop that summarises old turns when the conversation gets long, instead of feeding the model the whole history.

## 1. Setup

We re-import nb 01's tools, then **open a writable DuckDB connection** for the new memory table. Note: the writable connection cannot coexist with nb 01's read-only one, so we close that one first.

If you get a DuckDB lock error, shut down kernels of any other CTI 501 / 401 notebook before running this cell.

In [ ]:
import json, nbformat, ollama
from pathlib import Path
from datetime import datetime, timezone
import duckdb

nb = nbformat.read(Path("01_tools_not_agents.ipynb"), as_version=4)
for cell in nb.cells:
    if cell.cell_type == "code":
        exec(cell.source, globals())

# Replace nb 01's read-only connection with a writable one.
RO_CON.close()
RW_CON = duckdb.connect(str(DB_PATH), read_only=False)
RO_CON = RW_CON  # query tools from nb 01 use this name; rebind it

RW_CON.execute("""
CREATE TABLE IF NOT EXISTS agent_memory (
    note_id     INTEGER PRIMARY KEY,
    topic       TEXT,
    note        TEXT,
    related_id  TEXT,         -- url_id or alert_id this note is about, if any
    created_at  TIMESTAMP
);
""")
RW_CON.execute("CREATE SEQUENCE IF NOT EXISTS agent_memory_seq START 1;")
print("agent_memory table ready")

## 2. Two new tools — write and read agent memory

These look just like the tools in nb 01, but one of them mutates the database. Keep their behaviour narrow: write one note at a time, read by topic match. No bulk writes, no deletes — keeps things easy to reason about and impossible to accidentally wipe history.

In [ ]:
REMEMBER_NOTE_SCHEMA = {
    "name": "remember_note",
    "description": (
        "Save a short note to long-term memory so it can be recalled in future "
        "conversations. Use for facts you want to keep, like 'I read url_id X and "
        "it was about Y'."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "topic": {"type": "string", "description": "Short topic / tag for the note."},
            "note": {"type": "string", "description": "The note text. Keep under 300 chars."},
            "related_id": {"type": "string",
                            "description": "Optional url_id or alert_id this note is about."},
        },
        "required": ["topic", "note"],
    },
}

def remember_note(topic, note, related_id=None):
    if not topic or not note:
        return {"error": "topic and note are required"}
    note = note[:500]
    nid = RW_CON.execute("SELECT nextval('agent_memory_seq')").fetchone()[0]
    RW_CON.execute(
        "INSERT INTO agent_memory VALUES (?, ?, ?, ?, ?)",
        [nid, topic[:100], note, related_id, datetime.now(timezone.utc)],
    )
    return {"ok": True, "note_id": nid}

RECALL_NOTES_SCHEMA = {
    "name": "recall_notes",
    "description": (
        "Recall recent notes from long-term memory. Optionally filter by topic substring."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "topic_contains": {"type": "string",
                                "description": "Case-insensitive substring to filter topics."},
            "limit": {"type": "integer", "description": "Max notes. Default 5, max 15."},
        },
        "required": [],
    },
}

def recall_notes(topic_contains=None, limit=5):
    limit = max(1, min(int(limit or 5), 15))
    if topic_contains:
        rows = RW_CON.execute("""
            SELECT note_id, topic, note, related_id, created_at
            FROM agent_memory
            WHERE LOWER(topic) LIKE LOWER(?)
            ORDER BY created_at DESC
            LIMIT ?
        """, [f"%{topic_contains}%", limit]).fetchall()
    else:
        rows = RW_CON.execute("""
            SELECT note_id, topic, note, related_id, created_at
            FROM agent_memory ORDER BY created_at DESC LIMIT ?
        """, [limit]).fetchall()
    return {
        "count": len(rows),
        "notes": [
            {"note_id": r[0], "topic": r[1], "note": r[2],
             "related_id": r[3], "created_at": str(r[4])}
            for r in rows
        ],
    }

# Extend the registry with the two new tools.
TOOL_SCHEMAS.append({"type": "function", "function": REMEMBER_NOTE_SCHEMA})
TOOL_SCHEMAS.append({"type": "function", "function": RECALL_NOTES_SCHEMA})
TOOL_FUNCTIONS["remember_note"] = remember_note
TOOL_FUNCTIONS["recall_notes"] = recall_notes

print("tools now:", [t['function']['name'] for t in TOOL_SCHEMAS])

## 3. A loop that uses memory + a rolling summary

Two changes vs nb 03:

1. **Updated system prompt** that tells the model two new tools exist and *when* to use them.
2. **Rolling summary** — when the conversation history grows past ~12 messages, replace the oldest tool exchanges with a one-line summary, so context stays small.

In [ ]:
MODEL = "llama3.1:8b"
MAX_STEPS = 4
MAX_TOOL_OUTPUT_CHARS = 1500
MAX_TOTAL_TOOL_CALLS = 6
MAX_HISTORY_MESSAGES = 12

SYSTEM_PROMPT_MEM = (
    "You are a CTI analyst assistant with tools to query a corpus, query a public "
    "ransomware feed, and read/write a small long-term memory.\n\n"
    "Memory rules:\n"
    "- Before answering a question, consider calling recall_notes to check for prior context.\n"
    "- After completing a non-trivial task, consider calling remember_note to save a 1-line summary.\n"
    "- Do not save chit-chat or trivial facts. Only save things you'd want next session.\n\n"
    "Loop rules:\n"
    "- Stop calling tools as soon as you can answer.\n"
    "- Do not call the same tool with the same arguments twice in a row.\n"
    "- Keep answers short and factual."
)

def truncate_for_model(payload, max_chars=MAX_TOOL_OUTPUT_CHARS):
    s = json.dumps(payload)
    return s if len(s) <= max_chars else s[:max_chars] + f"... [truncated, total {len(s)} chars]"

def compress_history(messages):
    """Replace the oldest non-system, non-final messages with a one-line summary.
    Keeps the system prompt, the most recent user turn, and the last 6 messages intact.
    """
    if len(messages) <= MAX_HISTORY_MESSAGES:
        return messages
    head = [messages[0]]                     # system
    tail = messages[-6:]                     # most recent exchange
    middle = messages[1:-6]
    summary_lines = []
    for m in middle:
        if m["role"] == "tool":
            summary_lines.append(f"[tool {m.get('name','?')}] {(m.get('content') or '')[:80]}")
        elif m["role"] == "assistant" and m.get("tool_calls"):
            calls = ", ".join(tc["function"]["name"] for tc in m["tool_calls"])
            summary_lines.append(f"[assistant called {calls}]")
        elif m["role"] == "user":
            summary_lines.append(f"[user] {m['content'][:80]}")
    summary = "Earlier in this conversation:\n" + "\n".join(summary_lines[-12:])
    return head + [{"role": "system", "content": summary}] + tail

def run_agent_with_memory(messages, *, model=MODEL, verbose=True):
    """Run a planning loop on a *given* messages list. Caller owns the history
    (so multi-turn conversations are just repeated calls with the list extended).
    Returns (final_text, updated_messages)."""
    total_tool_calls = 0

    for step in range(1, MAX_STEPS + 1):
        if verbose:
            print(f"\n=== step {step} ===")

        messages = compress_history(messages)
        resp = ollama.chat(model=model, messages=messages, tools=TOOL_SCHEMAS,
                           options={"temperature": 0.2, "num_ctx": 4096})
        msg = resp["message"]
        messages = messages + [msg]

        tool_calls = msg.get("tool_calls") or []
        if not tool_calls:
            if verbose:
                print("  (final answer)")
                print(" ", (msg.get("content") or "")[:400])
            return msg.get("content") or "", messages

        for tc in tool_calls:
            fn_name = tc["function"]["name"]
            fn_args = tc["function"].get("arguments") or {}
            if isinstance(fn_args, str):
                try: fn_args = json.loads(fn_args)
                except json.JSONDecodeError: fn_args = {}

            total_tool_calls += 1
            if total_tool_calls > MAX_TOTAL_TOOL_CALLS:
                if verbose: print(f"  [cap: total tool calls > {MAX_TOTAL_TOOL_CALLS}]")
                return "(stopped: tool-call cap)", messages

            result = dispatch_tool_call(fn_name, fn_args)
            content = truncate_for_model(result)
            messages.append({"role": "tool", "name": fn_name, "content": content})
            if verbose:
                print(f"  -> {fn_name}({fn_args})")
                print(f"     {content[:200]}")

    if verbose: print("\n[cap: reached MAX_STEPS]")
    return "(stopped: step cap)", messages

## 4. Multi-turn demo

Three turns sharing one history list. After turn 2 the agent should have written a note; in turn 3 we see whether `recall_notes` retrieves it.

In [ ]:
history = [{"role": "system", "content": SYSTEM_PROMPT_MEM}]

history.append({"role": "user",
                "content": "Find one document labeled 'Hacking' and tell me what it's about. Save a one-line note about what you found."})
answer1, history = run_agent_with_memory(history)
print("\n=== FINAL 1 ===\n", answer1)

In [ ]:
history.append({"role": "user", "content": "How many ransomware victims came in over the last 30 days?"})
answer2, history = run_agent_with_memory(history)
print("\n=== FINAL 2 ===\n", answer2)

In [ ]:
history.append({"role": "user",
                "content": "What did you previously tell me about a Hacking-labeled document? Use your memory."})
answer3, history = run_agent_with_memory(history)
print("\n=== FINAL 3 ===\n", answer3)

# What did the agent actually save?
print("\nNotes in DB:")
for n in recall_notes(limit=10)["notes"]:
    print(" ", n)

## 5. What's next

nb 05 splits the work between **two** agents that talk via DuckDB rather than via shared messages: a *collector* that decides what to fetch, and an *analyst* that decides what to alert on. Sequential, not concurrent — easier to reason about and easier to debug than parallel multi-agent setups.

## Recap

Memory is two things, kept separate:

- **Short-term** = the conversation. Compressed when it grows.
- **Long-term** = a database. Read and written through tools.

The model is stateless. Your tools and the database hold the state. This is the pattern that scales.